# Image Proximity to FloodNet Sensors

This notebook analyzes the spatial proximity of dashcam images to FloodNet sensors reported in `aggregation/flooding/static/current_floodnet_sensors.csv`.

We use different spatial distance thresholds to count how many images are 'very close' to the sensors.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
import numpy as np
import os
import sys
from pathlib import Path
import shutil

# Add root to path for constants and helpers
root_dir = Path("../../")
sys.path.append(str(root_dir))

from notebooks.for_revisions.constants import WGS, PROJ

## Load Data

In [2]:
# Load FloodNet sensors
sensors_path = root_dir / "aggregation/flooding/static/sep23_floodnet_sensor_coordinates.csv"
sensors_df = pd.read_csv(sensors_path)
print(f"Loaded {len(sensors_df)} sensors.")

# Optional: Filter for active sensors if needed
# sensors_df = sensors_df[sensors_df['sensor_status'] == 'good']

# Create GeoDataFrame for sensors
sensors_gdf = gpd.GeoDataFrame(
    sensors_df, 
    geometry=gpd.points_from_xy(sensors_df.lon, sensors_df.lat),
    crs=WGS
)
# Project to NYC state plane (feet)
sensors_gdf = sensors_gdf.to_crs(PROJ)
sensors_gdf.head()

Loaded 75 sensors.


,deployment_id,lat,lon,geometry
0,asleep_apricot_bedbug,40.611330,-74.151058,POINT (942308.881 162036.639)
1,barely_brave_caiman,40.844349,-73.785508,POINT (1043595.684 246969.172)
2,barely_hot_aphid,40.660232,-73.936183,POINT (1001955.701 179823.05)
3,big_pink_elephant,40.655907,-73.828642,POINT (1031795.683 178287.678)
4,blue_eyed_tiger,40.700860,-73.954595,POINT (996839.79 194621.987)


In [3]:
# Load image metadata
# Note: md.csv is large, so we only load necessary columns
images_path = root_dir / "data/processed/md.csv"
# Loading h3_index_res06 and captured_at to construct local paths
images_metadata = pd.read_csv(images_path, usecols=['frame_id', 'gps_info.longitude', 'gps_info.latitude', 'h3_index_res06', 'captured_at'])
print(f"Loaded {len(images_metadata)} image metadata rows.")

# Convert captured_at (ms) to YYYY-MM-DD
images_metadata['date_str'] = pd.to_datetime(images_metadata['captured_at'], unit='ms').dt.strftime('%Y-%m-%d')

# Drop rows with missing GPS info
images_metadata = images_metadata.dropna(subset=['gps_info.latitude', 'gps_info.longitude'])
print(f"Remaining images with GPS: {len(images_metadata)}")

Loaded 926212 image metadata rows.
Remaining images with GPS: 926212


In [4]:
# Create GeoDataFrame for images
images_gdf = gpd.GeoDataFrame(
    images_metadata, 
    geometry=gpd.points_from_xy(images_metadata['gps_info.longitude'], images_metadata['gps_info.latitude']),
    crs=WGS
)
# Project to NYC state plane (feet)
images_gdf = images_gdf.to_crs(PROJ)
images_gdf.head()

,frame_id,captured_at,gps_info.longitude,gps_info.latitude,h3_index_res06,date_str,geometry
0,69fadc9f8ba1fca3f8ef8e0ac068cf45,1695989908491,-73.803642,40.865300,604222321527357439,2023-09-29,POINT (1038561.264 254590.639)
1,64131d7b3fdd56d6465b3b15f4f476a0,1695989402704,-73.803653,40.865214,604222321527357439,2023-09-29,POINT (1038558.292 254559.299)
2,afed5ce64c907520c2f12c3a9011f917,1696012995177,-73.826702,40.860566,604222321527357439,2023-09-29,POINT (1032186.449 252852.404)
3,b2d59019fa71d6f347d6f55e044cea8c,1696012986355,-73.826293,40.860859,604222321527357439,2023-09-29,POINT (1032299.372 252959.379)
4,40f12ae718ead40bfbf42446c1747cb4,1696012976553,-73.826118,40.861235,604222321527357439,2023-09-29,POINT (1032347.507 253096.467)


## Proximity Analysis

We use a cKDTree for efficient spatial lookups.

In [5]:
# Build tree from images
image_coords = np.array(list(zip(images_gdf.geometry.x, images_gdf.geometry.y)))
image_tree = cKDTree(image_coords)

# Define distance thresholds (in meters)
thresholds_m = [5, 10, 20, 50, 100, 250, 500]
meters_to_feet = 3.28084

results = []

for dist_m in thresholds_m:
    dist_ft = dist_m * meters_to_feet
    
    # query_ball_point returns indices of points within distance
    sensor_coords = np.array(list(zip(sensors_gdf.geometry.x, sensors_gdf.geometry.y)))
    indices = image_tree.query_ball_point(sensor_coords, dist_ft)
    
    # Unique images close to ANY sensor
    unique_image_indices = set()
    for idx_list in indices:
        unique_image_indices.update(idx_list)
    
    num_images = len(unique_image_indices)
    
    results.append({
        'threshold_meters': dist_m,
        'threshold_feet': dist_ft,
        'num_images_close': num_images,
        'pct_of_total': (num_images / len(images_gdf)) * 100
    })

results_df = pd.DataFrame(results)
results_df

,threshold_meters,threshold_feet,num_images_close,pct_of_total
0,5,16.4042,1,0.000108
1,10,32.8084,58,0.006262
2,20,65.6168,212,0.022889
3,50,164.0420,833,0.089936
4,100,328.0840,2429,0.262251
5,250,820.2100,11741,1.267636
6,500,1640.4200,37283,4.025320


## Per-Sensor Stats

Let's see which sensors have the most images nearby at a 50m threshold.

In [6]:
target_dist_m = 15
target_dist_ft = target_dist_m * meters_to_feet

sensor_coords = np.array(list(zip(sensors_gdf.geometry.x, sensors_gdf.geometry.y)))
indices = image_tree.query_ball_point(sensor_coords, target_dist_ft)

sensors_gdf['nearby_images'] = [len(idx_list) for idx_list in indices]
top_sensors = sensors_gdf.sort_values('nearby_images', ascending=False)[['deployment_id', 'nearby_images']]
top_sensors.head(20)

,deployment_id,nearby_images
52,simply_half_monkey,11
32,light_maroon_penguin,10
53,slowly_fast_sawfly,8
42,mildly_loving_doe,8
73,widely_whole_tarpon,7
67,weekly_poetic_guinea,6
40,mean_flying_fish,6
4,blue_eyed_tiger,6
20,firmly_needed_heron,5
47,overly_heroic_squid,5


In [7]:
print(f"Total sensors with at least one image within {target_dist_m}m: {(sensors_gdf['nearby_images'] > 0).sum()} / {len(sensors_gdf)}")

Total sensors with at least one image within 15m: 36 / 75


## Save Images for Visual Inspection

This cell copies the candidate images from the local Nexar data folder for each sensor.

In [8]:
from tqdm.auto import tqdm

# Create output folder
output_base = root_dir / "notebooks/for_revisions/sensor_proximity_images"
output_base.mkdir(exist_ok=True, parents=True)

nexar_data_root = Path("/share/ju/nexar_data/2023")

print(f"Saving images to {output_base}")

# Using target_dist_ft and indices from above (calculated in cell 9)
total_copied = 0

for i, sensor_indices in enumerate(tqdm(indices, desc="Sensors")):
    if len(sensor_indices) == 0:
        continue
    
    # Clean sensor name for folder
    sensor_name = sensors_gdf.iloc[i]['deployment_id'].replace("/", "_").replace(" ", "_").replace("\\", "_")
    sensor_folder = output_base / sensor_name
    sensor_folder.mkdir(exist_ok=True)
    
    for idx in sensor_indices:
        img_row = images_gdf.iloc[idx]
        img_id = img_row['frame_id']
        date_str = img_row['date_str']
        h3_06 = str(int(img_row['h3_index_res06'])) # Ensure it's string integer
        
        # Construct local path: /share/ju/nexar_data/2023/{date}/{h3_index_res06}/frames/{frame_id}.jpg
        source_path = nexar_data_root / date_str / h3_06 / "frames" / f"{img_id}.jpg"
        target_path = sensor_folder / f"{img_id}.jpg"
        
        if not target_path.exists():
            if source_path.exists():
                try:
                    shutil.copy(source_path, target_path)
                    total_copied += 1
                except Exception as e:
                    pass
            else:
                # If full frame doesn't exist, this might be a reason for failure
                pass

print(f"Finished. Copied {total_copied} images for visual inspection.")

/share/ju/matt/bayflood/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saving images to ../../notebooks/for_revisions/sensor_proximity_images


Sensors: 100%|██████████| 75/75 [00:10<00:00,  7.07it/s]

Finished. Copied 134 images for visual inspection.


## Export Nearby Image Metadata

Create a flat DataFrame of all images within 15m of a sensor, including sensor IDs and save to CSV.

In [10]:
# Create list of image-sensor pairs
pairs = []
for i, sensor_indices in enumerate(indices):
    if len(sensor_indices) > 0:
        sensor_id = sensors_gdf.iloc[i]['deployment_id']
        for idx in sensor_indices:
            pairs.append({'image_idx': idx, 'sensor_id': sensor_id})

# Convert to DataFrame and merge with metadata
nearby_df = pd.DataFrame(pairs)
nearby_metadata = nearby_df.merge(
    images_gdf.drop(columns=['geometry']).reset_index().rename(columns={'index': 'image_idx'}),
    on='image_idx'
)

# Add image_path column
nexar_data_root = "/share/ju/nexar_data/2023"
nearby_metadata['image_path'] = nearby_metadata.apply(
    lambda row: f"{nexar_data_root}/{row['date_str']}/{int(row['h3_index_res06'])}/frames/{row['frame_id']}.jpg", 
    axis=1
)

# Save to CSV
output_csv = root_dir / "notebooks/for_revisions/nearby_sensor_images_metadata.csv"
nearby_metadata.to_csv(output_csv, index=False)
print(f"Saved {len(nearby_metadata)} image-sensor records to {output_csv}")

nearby_metadata.head()

Saved 135 image-sensor records to ../../notebooks/for_revisions/nearby_sensor_images_metadata.csv


,image_idx,sensor_id,frame_id,captured_at,gps_info.longitude,gps_info.latitude,h3_index_res06,date_str,image_path
0,304740,big_pink_elephant,7be36d0e0df50f2193bb6ccd814a063e,1696038972603,-73.828545,40.655864,604222337365049343,2023-09-30,/share/ju/nexar_data/2023/2023-09-30/604222337...
1,304742,big_pink_elephant,9d3124e1bef05540fb37220d88ec75a5,1695987807569,-73.828762,40.655870,604222337365049343,2023-09-29,/share/ju/nexar_data/2023/2023-09-29/604222337...
2,653814,blue_eyed_tiger,8c92a379cb7bd8a944d352ff532d063f,1696024492290,-73.954460,40.700816,604222325151236095,2023-09-29,/share/ju/nexar_data/2023/2023-09-29/604222325...
3,653818,blue_eyed_tiger,220cd0557a956769dea5dcfb2d3ff244,1696023274031,-73.954529,40.700878,604222325151236095,2023-09-29,/share/ju/nexar_data/2023/2023-09-29/604222325...
4,653826,blue_eyed_tiger,3613f4b2b22ebb8fa4bbbc7e328dda34,1696019538488,-73.954491,40.700844,604222325151236095,2023-09-29,/share/ju/nexar_data/2023/2023-09-29/604222325...
